# Orquestación Automática de Respuesta a Incidentes de Seguridad

## Introducción a la Automatización de Respuesta

La IA puede automatizar decisiones críticas en el ciclo de respuesta a incidentes:
**detección → análisis → clasificación → respuesta → logging**

Este cuaderno implementa:
- **Triaje automático**: clasificar incidentes por severidad (SVM)
- **Orquestación de respuestas**: Motor de decisión para ejecutar acciones automatizadas
- **Logging y análisis post-incidente** para auditoría y mejora continua


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from dataclasses import dataclass
from enum import IntEnum

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("Importaciones OK")

## 1. Dataset de incidentes de seguridad

Características del incidente:
- `hosts_afectados` — nº de sistemas comprometidos
- `bytes_exfiltrados` — volumen de datos exfiltrados (bytes)
- `duracion_seg` — duración del incidente en segundos
- `tipo_evento` — tipo de evento (0-4), codificado numéricamente
- `privilegios_elevados` — si se usaron privilegios de administrador (0/1)
- `accesos_externos` — conexiones a IPs externas sospechosas
- `intentos_fallidos` — intentos de autenticación fallidos previos

In [ ]:
def generar_dataset_incidentes(n=1500, seed=42):
    rng = np.random.default_rng(seed)

    # Características base
    hosts      = rng.integers(1, 50,    size=n)
    exfil      = rng.exponential(10_000, size=n)
    duracion   = rng.exponential(60,     size=n)
    tipo       = rng.integers(0, 5,     size=n)
    privilegios = rng.integers(0, 2,    size=n)
    externos   = rng.integers(0, 20,    size=n)
    fallidos   = rng.integers(0, 100,   size=n)

    # Etiqueta de severidad basada en reglas heurísticas + ruido
    score = (
        0.3 * (hosts / hosts.max()) +
        0.3 * (exfil / exfil.max()) +
        0.2 * privilegios +
        0.1 * (externos / externos.max()) +
        0.1 * (fallidos / fallidos.max()) +
        rng.normal(0, 0.05, size=n)    # ruido
    )
    labels = np.digitize(score, bins=[0.25, 0.5, 0.75]) # 0,1,2,3

    return pd.DataFrame({
        "hosts_afectados":    hosts,
        "bytes_exfiltrados":  exfil,
        "duracion_seg":       duracion,
        "tipo_evento":        tipo,
        "privilegios_elevados": privilegios,
        "accesos_externos":   externos,
        "intentos_fallidos":  fallidos,
        "severity":           labels,
    })

df = generar_dataset_incidentes()
etiquetas = ["Bajo", "Medio", "Alto", "Crítico"]
print(f"Dataset: {df.shape}")
print("Distribución de severidad:")
print(df["severity"].map(dict(enumerate(etiquetas))).value_counts().sort_index())
df.head()

## 2. Análisis exploratorio

In [ ]:
FEATURES = ["hosts_afectados", "bytes_exfiltrados", "duracion_seg",
            "tipo_evento", "privilegios_elevados", "accesos_externos", "intentos_fallidos"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
palette = ["#4CAF50", "#FFC107", "#FF5722", "#B71C1C"]

for i, col in enumerate(FEATURES):
    for sev in range(4):
        datos = df[df.severity == sev][col]
        axes[i].hist(datos, bins=20, alpha=0.5, color=palette[sev],
                     label=etiquetas[sev], density=True)
    axes[i].set_title(col)
    if i == 0:
        axes[i].legend(fontsize=7)

axes[-1].axis("off")
plt.suptitle("Distribución de características por nivel de severidad", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Entrenamiento del modelo SVM con búsqueda de hiperparámetros

In [ ]:
X = df[FEATURES]
y = df["severity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Pipeline: escalado + SVM
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",    SVC(kernel="rbf", probability=True, random_state=SEED))
])

# Búsqueda de hiperparámetros con GridSearchCV
param_grid = {
    "svm__C":     [0.1, 1.0, 10.0],
    "svm__gamma": ["scale", "auto"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv,
    scoring="f1_weighted", n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nMejores parámetros: {grid_search.best_params_}")
print(f"Mejor F1 (CV):      {grid_search.best_score_:.4f}")

In [ ]:
# Evaluación en conjunto de prueba
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred, target_names=etiquetas, zero_division=0))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(cm, display_labels=etiquetas).plot(ax=ax, cmap="OrRd", colorbar=False)
ax.set_title("Matriz de Confusión — Triaje de Incidentes (SVM)")
plt.tight_layout()
plt.show()

## 4. Motor de orquestación de respuestas

Dado el nivel de severidad predicho, el motor ejecuta acciones automáticas
escalonadas: registrar → notificar → bloquear → aislar.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S"
)

class Severidad(IntEnum):
    BAJO    = 0
    MEDIO   = 1
    ALTO    = 2
    CRITICO = 3

@dataclass
class Incidente:
    id_sistema  : str
    ip_origen   : str
    severidad   : Severidad
    descripcion : str

# Acciones simuladas (en producción llamarían a APIs reales)
def registrar(inc):
    logging.info(f"[REGISTRO] {inc.id_sistema} — {inc.descripcion}")

def notificar_equipo(inc):
    logging.warning(f"[NOTIFICACIÓN] Alerta enviada al equipo CSIRT — "
                    f"Sistema: {inc.id_sistema}, IP: {inc.ip_origen}")

def bloquear_ip(inc):
    logging.warning(f"[FIREWALL] IP {inc.ip_origen} bloqueada en perímetro")

def aislar_sistema(inc):
    logging.critical(f"[AISLAMIENTO] Sistema {inc.id_sistema} desconectado de la red")

def orquestar_respuesta(inc: Incidente) -> list:
    """Ejecuta acciones de respuesta según la severidad del incidente."""
    acciones_ejecutadas = []

    if inc.severidad >= Severidad.BAJO:
        registrar(inc)
        acciones_ejecutadas.append("registrar")

    if inc.severidad >= Severidad.MEDIO:
        notificar_equipo(inc)
        acciones_ejecutadas.append("notificar")

    if inc.severidad >= Severidad.ALTO:
        bloquear_ip(inc)
        acciones_ejecutadas.append("bloquear_ip")

    if inc.severidad >= Severidad.CRITICO:
        aislar_sistema(inc)
        acciones_ejecutadas.append("aislar_sistema")

    return acciones_ejecutadas

print("Motor de orquestación definido.")

In [ ]:
# ── Simulación: clasificar un nuevo incidente y responder ───────────────
nuevos_incidentes_raw = pd.DataFrame([
    {"hosts_afectados": 1,  "bytes_exfiltrados": 1_000,   "duracion_seg": 5,
     "tipo_evento": 1, "privilegios_elevados": 0, "accesos_externos": 0, "intentos_fallidos": 2},
    {"hosts_afectados": 5,  "bytes_exfiltrados": 150_000,  "duracion_seg": 120,
     "tipo_evento": 3, "privilegios_elevados": 1, "accesos_externos": 5, "intentos_fallidos": 50},
    {"hosts_afectados": 20, "bytes_exfiltrados": 5_000_000,"duracion_seg": 3600,
     "tipo_evento": 4, "privilegios_elevados": 1, "accesos_externos": 15, "intentos_fallidos": 99},
])

ids    = ["SRV-DEV-01", "SRV-PROD-05", "SRV-DB-MASTER"]
ips    = ["10.0.1.15",  "192.168.5.30", "45.33.32.156"]
descs  = ["Escaneo de puertos",
          "Movimiento lateral detectado",
          "Exfiltración masiva de base de datos"]

severidades_pred = best_model.predict(nuevos_incidentes_raw)

print("=" * 60)
for i, sev_pred in enumerate(severidades_pred):
    inc = Incidente(
        id_sistema  = ids[i],
        ip_origen   = ips[i],
        severidad   = Severidad(sev_pred),
        descripcion = descs[i]
    )
    print(f"\n[INCIDENTE] {inc.id_sistema} → Severidad: {inc.severidad.name}")
    acciones = orquestar_respuesta(inc)
    print(f"  Acciones ejecutadas: {acciones}")

print("\n✓ Cuaderno 4 completado.")